#Load Libraries


In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import mutual_info_classif
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from matplotlib.colors import LogNorm

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = '/content/drive/MyDrive/zeek24data/zeekdatasets'

# 1. Grab all the CSV files from your defined path
all_files = glob.glob(os.path.join(path, "*.csv"))

all_files = [file for file in all_files if not file.endswith("zeekdata_combined.csv")]

# 2. Create an empty list to store each dataframe temporarily
df_list = []

# 3. Loop through each file, read it into pandas, and add it to the list
for file in all_files:
    print(f"Loading {os.path.basename(file)}...")
    df = pd.read_csv(file)
    df_list.append(df)

# 4. Concatenate them vertically (axis=0) aligning by column names
# ignore_index=True ensures the index resets smoothly from 0 to the end
combined_df = pd.concat(df_list, axis=0, ignore_index=True)

print(combined_df['label_tactic'].value_counts())

# 5. Verification checks
print("\n--- Combination Complete ---")
print(f"Total rows in combined dataset: {combined_df.shape[0]}")
print(f"Total columns: {combined_df.shape[1]}")

# Display the class distribution to ensure labels combined correctly
# Note: Change 'label' if your target column has a different name (e.g., 'Label', 'class')
if 'label' in combined_df.columns:
    print("\nCombined Label Distribution:")
    print(combined_df['label'].value_counts())
else:
    print("\nColumns available:", combined_df.columns.tolist())


  # 1. Define the name of your new combined file
# This will save it in the same 'path' directory you defined earlier
output_filename = "zeekdata_combined.csv"
output_filepath = os.path.join(path, output_filename)

# 2. Export the dataframe to CSV
print(f"Exporting combined dataset to {output_filepath}...")

# index=False is crucial here! It prevents pandas from exporting the row numbers
# as a new, meaningless column that would mess up your machine learning models later.
combined_df.to_csv(output_filepath, index=False)

# 3. Confirmation
print(f"Export successful! File saved as: {output_filename}")




Loading zeekdata24fall.csv...
Loading zeekdata22fall.csv...
Loading zeekdata24.csv...


/tmp/ipykernel_5463/2916303948.py:14: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


label_tactic
none                     138167
Discovery                 67585
Credential Access         43495
Resource Development      13644
Reconnaissance            10690
Defense Evasion             873
Initial Access              617
Privilege Escalation        547
Persistence                 336
Priviledge Escalation       326
Lateral Movement             40
Execution                    32
Exfiltration                 23
Command and Control          20
Collection                    1
Name: count, dtype: int64

--- Combination Complete ---
Total rows in combined dataset: 276396
Total columns: 23

Columns available: ['community_id', 'conn_state', 'duration', 'history', 'src_ip_zeek', 'src_port_zeek', 'dest_ip_zeek', 'dest_port_zeek', 'local_orig', 'local_resp', 'missed_bytes', 'orig_bytes', 'orig_ip_bytes', 'orig_pkts', 'proto', 'resp_bytes', 'resp_ip_bytes', 'resp_pkts', 'service', 'ts', 'uid', 'datetime', 'label_tactic']
Exporting combined dataset to /content/drive/MyDrive/zeek24da

In [ ]:
# 1. Define the exact names of the 4 classes you want to keep
# (Make sure spelling and capitalization perfectly match the output from earlier!)
top_4_classes = ['none', 'Discovery', 'Credential Access', 'Resource Development']

# 2. Filter the dataframe to ONLY include rows where 'label_tactic' is in your top 4 list
combined_df = combined_df[combined_df['label_tactic'].isin(top_4_classes)].copy()

# 3. Reset the index so the row numbers remain continuous after dropping thousands of rows
combined_df.reset_index(drop=True, inplace=True)

# 4. Verify the new distribution to ensure it worked perfectly
print("\n--- NEW Class Distribution (Top 4 Classes Only) ---")
print(combined_df['label_tactic'].value_counts())

# Print the new total size of your dataset
print(f"\nNew total rows after dropping minority classes: {combined_df.shape[0]}")


--- NEW Class Distribution (Top 4 Classes Only) ---
label_tactic
none                    138167
Discovery                67585
Credential Access        43495
Resource Development     13644
Name: count, dtype: int64

New total rows after dropping minority classes: 262891


#DISTRIBUTION

In [ ]:
if 'label_tactic' in combined_df.columns:
    print("\n--- Class Distribution (label_tactic) ---")

    # Calculate the counts for each class
    distribution = combined_df['label_tactic'].value_counts()
    print(distribution)

    # Optional: Print the distribution as percentages to see the imbalance clearly
    print("\n--- Percentage Distribution ---")
    percentages = combined_df['label_tactic'].value_counts(normalize=True) * 100
    print(percentages.round(4).astype(str) + '%')
else:
    print("\nError: 'label_tactic' column not found. Available columns are:")
    print(combined_df.columns.tolist())


--- Class Distribution (label_tactic) ---
label_tactic
none                    138167
Discovery                67585
Credential Access        43495
Resource Development     13644
Name: count, dtype: int64

--- Percentage Distribution ---
label_tactic
none                    52.5568%
Discovery               25.7084%
Credential Access       16.5449%
Resource Development       5.19%
Name: proportion, dtype: object
